# AI vs Human Writing Detection Project

## 1. Background
Natural Language Processing (NLP) focuses on enabling computers to understand and process human language. One important area of NLP is text classification. In this project, we classify whether a piece of text was written by a human or generated by an AI model. This is important because large language models are widely accessible and raise concerns in education, academic integrity, and online content authenticity.

## 2. Topic / Problem
The goal of this project is to classify text as human-written or AI-generated. We design and evaluate both classical machine learning and transformer-based deep learning approaches for this binary classification task.

## 3. Dataset
We use the **ai-and-human-text-dataset** from Kaggle:
- Source: https://www.kaggle.com/datasets/hasanyiitakbulut/ai-and-human-text-dataset
- Input example: text snippets
- Output label: `0 = human`, `1 = AI`
- Size: approximately 6,069 samples

This notebook assumes the dataset CSV is in `data/` under the project root.


In [ ]:
# If dependencies are missing, install them once in your environment:
# %pip install pandas numpy scikit-learn nltk torch transformers

import os
import re
import glob
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


In [ ]:
# 4. Methods: Data Loading + Preprocessing

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'

# Common filename patterns for this dataset across downloads/shares.
CANDIDATE_PATTERNS = [
    '*ai*human*.csv',
    '*AI*Human*.csv',
    '*text*.csv',
    '*.csv',
]


def find_dataset_csv(data_dir: Path) -> Path:
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Expected data directory at {data_dir}. Create it and place the Kaggle CSV inside."
        )

    candidates = []
    for pattern in CANDIDATE_PATTERNS:
        candidates.extend(sorted(data_dir.glob(pattern)))

    # Keep order, remove duplicates
    seen = set()
    unique = []
    for path in candidates:
        p = str(path.resolve())
        if p not in seen:
            seen.add(p)
            unique.append(path)

    if not unique:
        raise FileNotFoundError(
            "No CSV files found in data/. Download from Kaggle and place the CSV in data/."
        )

    # Prefer a file that looks like AI-vs-human text data.
    for path in unique:
        name = path.name.lower()
        if 'ai' in name and 'human' in name:
            return path

    return unique[0]


def normalize_columns(df: pd.DataFrame):
    # Detect text column
    text_candidates = ['text', 'Text', 'content', 'sentence']
    text_col = next((c for c in text_candidates if c in df.columns), None)

    # Detect label column
    label_candidates = ['generated', 'label', 'Category', 'category', 'target']
    label_col = next((c for c in label_candidates if c in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(
            f"Could not detect text/label columns. Found columns: {list(df.columns)}"
        )

    out = df[[text_col, label_col]].copy()
    out = out.rename(columns={text_col: 'text', label_col: 'label'})

    # Map labels to {0,1} when needed.
    if out['label'].dtype == object:
        label_map = {
            'human': 0,
            'ai': 1,
            'ai-generated': 1,
            'generated': 1,
            'machine': 1,
            '0': 0,
            '1': 1,
        }
        out['label'] = out['label'].astype(str).str.strip().str.lower().map(label_map)

    out = out.dropna(subset=['text', 'label']).copy()
    out['label'] = out['label'].astype(int)
    out = out[out['label'].isin([0, 1])]
    out['text'] = out['text'].astype(str)

    return out


def preprocess_text(text: str, stop_words_set: set) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    cleaned = [tok for tok in tokens if tok not in stop_words_set and len(tok) > 1]
    return ' '.join(cleaned)


csv_path = find_dataset_csv(DATA_DIR)
print(f'Using dataset file: {csv_path}')

raw_df = pd.read_csv(csv_path)
df = normalize_columns(raw_df)
print(f'Dataset size after cleanup: {len(df)}')
print('Label distribution:')
print(df['label'].value_counts(normalize=True).sort_index())

stop_words_set = set(stopwords.words('english'))
df['cleaned_text'] = df['text'].apply(lambda x: preprocess_text(x, stop_words_set))


In [ ]:
# Split once for fair comparison across models
X_train_clean, X_test_clean, y_train, y_test, X_train_raw, X_test_raw = train_test_split(
    df['cleaned_text'],
    df['label'],
    df['text'],
    test_size=0.2,
    random_state=SEED,
    stratify=df['label']
)

print('Train size:', len(X_train_clean), 'Test size:', len(X_test_clean))
print('Train label distribution:')
print(y_train.value_counts(normalize=True).sort_index())


## 4.1 Baseline Model: TF-IDF + Logistic Regression
We convert preprocessed text to TF-IDF vectors, then train a logistic regression classifier as the baseline machine learning model.


In [ ]:
def train_tfidf_baseline(X_train, X_test, y_train, y_test):
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=2000, random_state=SEED)
    model.fit(X_train_tfidf, y_train)

    pred = model.predict(X_test_tfidf)
    metrics = {
        'model': 'TF-IDF + LogisticRegression',
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    }

    return model, vectorizer, pred, metrics


baseline_model, tfidf_vectorizer, baseline_pred, baseline_metrics = train_tfidf_baseline(
    X_train_clean, X_test_clean, y_train, y_test
)

print('Baseline classification report:')
print(classification_report(y_test, baseline_pred, digits=4))
pd.DataFrame([baseline_metrics])


## 4.2 Advanced Model: BERT Fine-Tuning (Practical Subset)
For runtime feasibility in local environments, we fine-tune BERT on a controlled subset of the training data.

If `transformers` or `torch` are not available, this section will fail with a clear message. You can install dependencies and rerun.


In [ ]:
import torch

try:
    from transformers import (
        BertTokenizerFast,
        BertForSequenceClassification,
        Trainer,
        TrainingArguments,
    )
except Exception as e:
    raise ImportError(
        "Transformers dependencies missing. Install with: %pip install torch transformers"
    ) from e


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
    }


class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


BERT_TRAIN_SAMPLES = min(3000, len(X_train_raw))
BERT_TEST_SAMPLES = min(1000, len(X_test_raw))
MAX_LEN = 256

bert_train_df = pd.DataFrame({'text': X_train_raw.values, 'label': y_train.values}).sample(
    n=BERT_TRAIN_SAMPLES, random_state=SEED, stratify=y_train.values
)
bert_test_df = pd.DataFrame({'text': X_test_raw.values, 'label': y_test.values}).sample(
    n=BERT_TEST_SAMPLES, random_state=SEED, stratify=y_test.values
)

print(f'BERT train subset: {len(bert_train_df)} | test subset: {len(bert_test_df)}')

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

train_enc = tokenizer(
    bert_train_df['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
)

test_enc = tokenizer(
    bert_test_df['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
)

train_dataset = TextDataset(train_enc, bert_train_df['label'].tolist())
test_dataset = TextDataset(test_enc, bert_test_df['label'].tolist())

bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='artifacts/bert_outputs',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=25,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
bert_eval = trainer.evaluate()
bert_metrics = {
    'model': 'BERT (fine-tuned subset)',
    'accuracy': float(bert_eval.get('eval_accuracy', np.nan)),
    'precision': float(bert_eval.get('eval_precision', np.nan)),
    'recall': float(bert_eval.get('eval_recall', np.nan)),
    'f1': float(bert_eval.get('eval_f1', np.nan)),
}
pd.DataFrame([bert_metrics])


## 5. Evaluation
We evaluate with **accuracy, precision, recall, and F1-score** and compare baseline vs advanced model performance.


In [ ]:
results = [baseline_metrics]
if 'bert_metrics' in globals():
    results.append(bert_metrics)

results_df = pd.DataFrame(results)
results_df


In [ ]:
# Optional: single-text inference helper using the trained BERT model
# Run this only if the BERT section above was executed successfully.

import torch.nn.functional as F

def detect_ai_text(text: str):
    if 'bert_model' not in globals() or 'tokenizer' not in globals():
        raise RuntimeError('BERT model/tokenizer not available. Run the BERT training section first.')

    bert_model.eval()
    inputs = tokenizer(text, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
    device = bert_model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)

    probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
    pred = int(np.argmax(probs))
    confidence = float(probs[pred])
    label_map = {0: 'Human', 1: 'AI-Generated'}

    return {
        'prediction': label_map[pred],
        'confidence': round(confidence, 4),
        'prob_human': round(float(probs[0]), 4),
        'prob_ai': round(float(probs[1]), 4),
    }


sample_text = 'I wrote this paragraph after reading three research papers and summarizing their findings.'
print(detect_ai_text(sample_text))


## Conclusion
- The TF-IDF baseline provides a strong and efficient reference.
- BERT can capture richer context and may improve classification quality, but requires more compute.
- For deployment, choose the model based on the tradeoff between performance and runtime/cost constraints.
